# GAT-v4 | Phase 8B — Biomarker Interpretability (Approach 3: Within-Pathway Gene Importance)

**Fixes applied to your original cell** — see the note at the bottom of this cell for
what was wrong and why.

### What this notebook does
For each BH-significant pathway (`padj < 0.05`, from `pathway_cox_results.csv`):
1. Take the pathway embedding (64-dim mean of `h_gene` over the pathway's genes)
2. Compute `d(||pathway_embedding||^2) / d(input_features)` for the genes in that
   pathway, on a sample of high-risk test patients from one reference fold
3. Gradient magnitude per gene = how much that gene's input drives the pathway score
4. Rank genes within each pathway by mean gradient magnitude → biomarker candidates

---
### Bugs fixed from your original cell

1. **`NameError: gene_to_idx`** — the actual crash. `gene_to_idx` was built via
   `{g: i for i, g in enumerate(ppi_genes)}` **after** the loop that already used it
   to build `pathway_gene_map`. Fixed by building `ppi_genes`/`gene_to_idx` first
   (Cell 2 below), same as Phase 8's own Cell 2.
2. **`GATv4`, `pat_list`, `high_risk_pats`, `pat_to_sids`, `feat_all`, `clin_all`,
   `ei_r`, `ew_r` were all undefined.** This notebook has a fresh kernel (only 2 cells
   ran), so nothing from Phase 8's session carries over — every dependency has to be
   rebuilt from files on disk. Added as their own cells (3 and 4) below.
3. **`fold_sids_list` was reloaded from disk on every single patient iteration**
   (`json.load(open(...))` inside the innermost loop) — up to
   `n_significant_pathways × 30` redundant file reads. Now loaded once in Cell 4.
4. **`fold_sids_list.index(sid)` was an O(n) linear scan per patient** — replaced with
   a precomputed `sid_to_fold_idx` dict lookup (O(1)).
5. **CPU/CUDA device mismatch**: `path_idx_t = torch.tensor(path_idx, dtype=torch.long)`
   defaults to CPU, but `hg` (from the model) lives on `DEVICE` (`cuda`, per your Cell 1
   output). Indexing a CUDA tensor with a CPU index tensor raises a `RuntimeError` in
   PyTorch. Fixed by creating `path_idx_t` directly on `DEVICE`.
6. **Fragile leaf-tensor construction**: `feat_all[idx].to(DEVICE).requires_grad_(True)`
   only reliably produces a proper autograd leaf if `.to()` actually performs a device
   copy. Replaced with the always-safe `.detach().clone().requires_grad_(True)` pattern.
7. **Model parameter gradients were never cleared** between patients — `.eval()` mode
   disables dropout/batchnorm updates but does **not** stop `GATConv`/`LayerNorm`
   parameters from accumulating `.grad` on every `.backward()` call. Harmless for the
   *gene* gradients (those come from `x.grad`, a fresh leaf each iteration) but wastes
   memory over hundreds of backward passes. Added `model_grad.zero_grad(set_to_none=True)`
   before each backward call.
8. **`torch.load(..., weights_only=True)`** — Phase 7 saves checkpoints as
   `{'epoch', 'model_state', 'val_cindex', 'fold_key'}`, a mix of tensors and plain
   Python types. Phase 8's own (working) loop loads these with `weights_only=False`.
   Matched that here for consistency with the one pattern you know already works.
9. Guarded the top-genes printout against pathways where no test patient had usable
   feature data (`within_pathway_importance[pid]` empty) to avoid a `KeyError`/empty
   `min()`/`max()` on the sort.


---
## Cell 1 — Imports & Paths

Unchanged from your version.


In [1]:
import sys, json, yaml, hashlib, time
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from lifelines import CoxPHFitter
from tqdm.auto import tqdm

# ── PATHS ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script")
KIRC_ROOT   = PROJECT_ROOT / "KIRC"
DATA_PROC    = KIRC_ROOT / "data" / "processed"
DATA_SPLITS  = KIRC_ROOT / "data" / "splits"
CKPT_DIR     = KIRC_ROOT / "results" / "checkpointsibsLOG"
LOG_DIR      = KIRC_ROOT / "results" / "logs"
TABLE_DIR    = KIRC_ROOT / "results" / "tables"
AUDIT_DIR    = KIRC_ROOT / "audit"
# ─────────────────────────────────────────────────────────────────────────────

TABLE_DIR.mkdir(parents=True, exist_ok=True)

if str(KIRC_ROOT) not in sys.path:
    sys.path.insert(0, str(KIRC_ROOT))
from src.utils.logger import get_logger
from src.config import GAT_IN_DIM, GAT_HIDDEN, GAT_HEADS, GAT_MLP_HIDDEN, GAT_CLINICAL_DIM, GAT_DROPOUT

log    = get_logger('phase8_pathway')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\jiyad\Desktop\PROJECT\KIRC\script\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\jiyad\Desktop\PROJECT\KIRC\script\.venv\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "C:\Users\jiyad\Desktop\PROJECT\KIRC\script\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 807, in start

Device : cuda


---
## Cell 2 — Load Gene Index, Reactome Pathways, and Significant Pathways

Same source data Phase 8 used (`ppi_gene_index.csv`, `reactome_filtered.csv`), plus the
Cox results Phase 8 already computed and saved (`pathway_cox_results.csv`).

**This is what fixes the original `NameError`** — `ppi_genes` / `gene_to_idx` now exist
*before* anything tries to use them.


In [2]:
# ── Gene index (PPI node ordering) — must exist before building pathway_gene_map ──
gene_idx_df = pd.read_csv(DATA_PROC / 'ppi_gene_index.csv')
ppi_genes   = gene_idx_df['gene_symbol'].tolist()
gene_to_idx = {g: i for i, g in enumerate(ppi_genes)}
n_nodes     = len(ppi_genes)
print(f"PPI nodes : {n_nodes:,}")

# ── Reactome filtered pathways ────────────────────────────────────────────────
reactome_df = pd.read_csv(DATA_PROC / 'reactome_filtered.csv')
print(f"Pathways in Reactome table : {len(reactome_df):,}")

# ── Significant pathways from Phase 8's Cox output ────────────────────────────
cox_df = pd.read_csv(TABLE_DIR / 'pathway_cox_results.csv')
sig_pathways = cox_df[cox_df['padj'] < 0.05].copy()
print(f"Significant pathways (BH q < 0.05) : {len(sig_pathways):,}")

# ── pathway_id -> PPI node indices / names (only pathways with >=3 PPI nodes) ─
MIN_PPI_NODES = 3

pathway_gene_map = {}
pathway_name_map = {}
for _, row in reactome_df.iterrows():
    pid   = row['reactome_id']
    genes = set(row['gene_symbols'].split('|'))
    idx   = [gene_to_idx[g] for g in genes if g in gene_to_idx]
    if len(idx) >= MIN_PPI_NODES:
        pathway_gene_map[pid] = idx
        pathway_name_map[pid] = row['pathway_name']

print(f"Pathways with >= {MIN_PPI_NODES} PPI nodes : {len(pathway_gene_map):,}")
n_sig_with_map = sig_pathways['reactome_id'].isin(pathway_gene_map).sum()
print(f"Significant pathways with a usable gene map : {n_sig_with_map} / {len(sig_pathways)}")


PPI nodes : 4,414
Pathways in Reactome table : 807
Significant pathways (BH q < 0.05) : 31
Pathways with >= 3 PPI nodes : 805
Significant pathways with a usable gene map : 31 / 31


---
## Cell 3 — Rebuild GATv4 Architecture + Identify High-Risk Patients

`GATv4` must be redefined in this session before any checkpoint can be loaded into it
(fresh kernel = no class definition carried over from Phase 8).

High-risk status is taken from Phase 8's saved `patient_risk_scores.csv`
(risk score aggregated across all 15 folds), split at the median — same convention
Phase 8 used in its own Cell 6.


In [3]:
# Re-define GATv4 (same architecture as Phase 7 / Phase 8)
class GATv4(nn.Module):
    def __init__(self, in_dim=GAT_IN_DIM, hidden=GAT_HIDDEN, heads=GAT_HEADS,
                 mlp_hidden=GAT_MLP_HIDDEN, clinical_dim=GAT_CLINICAL_DIM,
                 dropout=GAT_DROPOUT):
        super().__init__()
        self.dropout = dropout
        hd = hidden // heads
        self.gat1     = GATConv(in_dim,  hd, heads=heads, concat=True,
                                dropout=dropout, edge_dim=1, add_self_loops=True)
        self.norm1    = nn.LayerNorm(hidden)
        self.gat2     = GATConv(hidden, hd, heads=heads, concat=True,
                                dropout=dropout, edge_dim=1, add_self_loops=True)
        self.norm2    = nn.LayerNorm(hidden)
        self.residual = nn.Linear(in_dim, hidden, bias=False)
        self.risk_head = nn.Sequential(
            nn.Linear(hidden + clinical_dim, mlp_hidden), nn.ELU(),
            nn.Dropout(dropout), nn.Linear(mlp_hidden, 1)
        )

    def forward(self, x, edge_index, edge_weight, clinical):
        ew  = edge_weight.unsqueeze(-1)
        h   = F.elu(self.norm1(self.gat1(x, edge_index, edge_attr=ew)))
        h   = F.dropout(h, self.dropout, self.training)
        h2  = self.norm2(self.gat2(h, edge_index, edge_attr=ew))
        hg  = F.elu(h2 + self.residual(x))
        hg  = F.dropout(hg, self.dropout, self.training)
        zp  = hg.mean(dim=0)
        risk = self.risk_head(torch.cat([zp, clinical], dim=-1)).squeeze()
        return risk, hg


# ── High-risk patients (median split on aggregated risk score, all 15 folds) ──
risk_df = pd.read_csv(TABLE_DIR / 'patient_risk_scores.csv')
risk_median = float(risk_df['risk_score'].median())
high_risk_pats = set(risk_df.loc[risk_df['risk_score'] >= risk_median, 'patient_id'])
print(f"Risk median : {risk_median:.4f}")
print(f"High-risk patients (>= median) : {len(high_risk_pats):,} / {len(risk_df):,}")

# ── Sample alignment: patient <-> sample id maps ──────────────────────────────
align       = pd.read_csv(DATA_PROC / 'sample_alignment.csv')
align_tumor = align[(align['tissue_type'] == 'tumor') & align['include']]
pat_to_sids = align_tumor.groupby('patient_id')['sample_id'].apply(list).to_dict()
sid_to_pat  = dict(zip(align_tumor['sample_id'], align_tumor['patient_id']))
print(f"Patients with tumor samples : {len(pat_to_sids):,}")


Risk median : 0.0482
High-risk patients (>= median) : 246 / 491
Patients with tumor samples : 491


---
## Cell 4 — Load Reference Fold Data + Its Test-Set Patients

Gradients are computed against a **single representative fold's** checkpoint and graph
(`seed42_fold0`, matching your original cell) rather than all 15 — this is a
computational-tractability choice for an interpretability pass, not a full
cross-validated result.


In [4]:
REFERENCE_FOLD = 'seed42_fold0'
fold_dir = DATA_SPLITS / REFERENCE_FOLD

feat_all = torch.tensor(np.load(fold_dir / 'feature_tensor.npy'),  dtype=torch.float32)
clin_all = torch.tensor(np.load(fold_dir / 'clinical_tensor.npy'), dtype=torch.float32)
ei_r     = torch.tensor(np.load(fold_dir / 'edge_index.npy'),  dtype=torch.long).to(DEVICE)
ew_r     = torch.tensor(np.load(fold_dir / 'edge_weight.npy'), dtype=torch.float32).to(DEVICE)

with open(fold_dir / 'sample_ids.json') as f:
    fold_sids_list = json.load(f)          # loaded ONCE, not per-patient
with open(fold_dir / 'split_indices.json') as f:
    split_idx = json.load(f)

# O(1) sample_id -> row index lookup (replaces the per-patient .index() scan)
sid_to_fold_idx = {sid: i for i, sid in enumerate(fold_sids_list)}

test_sids = {fold_sids_list[i] for i in split_idx['test']}
pat_list  = sorted({sid_to_pat[s] for s in test_sids if s in sid_to_pat})

print(f"Reference fold      : {REFERENCE_FOLD}")
print(f"Feature tensor       : {tuple(feat_all.shape)}")
print(f"Test patients        : {len(pat_list):,}")
print(f"...of which high-risk: {sum(1 for p in pat_list if p in high_risk_pats):,}")


Reference fold      : seed42_fold0
Feature tensor       : (500, 4414, 3)
Test patients        : 99
...of which high-risk: 55


---
## Cell 5 — Approach 3: Within-Pathway Gene Importance (Fixed)

For each significant pathway: run a forward pass with gradient tracking on a sample of
high-risk test patients from `seed42_fold0`, backprop `||pathway_embedding||^2` to the
input features of that pathway's genes, and average the resulting gradient magnitude
per gene across patients. Higher = that gene's expression/CNV/features more strongly
drive the pathway's dysregulation score for high-risk patients.


In [5]:
N_PATIENTS_PER_PATHWAY = 30   # sample of high-risk test patients per pathway

# Reload checkpoint WITH gradient tracking on the model (eval mode, but params still
# require grad by default — that's fine, we zero their grads each step below)
ckpt_g = torch.load(CKPT_DIR / f'{REFERENCE_FOLD}_best_model.pt',
                     map_location=DEVICE, weights_only=False)
model_grad = GATv4().to(DEVICE)
model_grad.load_state_dict(ckpt_g['model_state'])
model_grad.eval()

within_pathway_importance = {}   # pathway_id -> {gene: mean_gradient}

test_pats_hr = [p for p in pat_list if p in high_risk_pats][:N_PATIENTS_PER_PATHWAY]
print(f"Using {len(test_pats_hr)} high-risk test patients from {REFERENCE_FOLD}")
print(f"Computing within-pathway gene importance for {len(sig_pathways)} pathways...")

for pid in tqdm(sig_pathways['reactome_id'].tolist(), desc='Pathways'):
    if pid not in pathway_gene_map:
        continue

    path_idx   = pathway_gene_map[pid]
    path_idx_t = torch.tensor(path_idx, dtype=torch.long, device=DEVICE)  # on DEVICE!
    gene_grads = defaultdict(list)

    for pat in test_pats_hr:
        sids_p = pat_to_sids.get(pat, [])
        sid = next((s for s in sids_p if s in sid_to_fold_idx), None)
        if sid is None:
            continue
        idx_in_fold = sid_to_fold_idx[sid]   # O(1) lookup, not list.index()

        # .detach().clone() guarantees a proper autograd leaf regardless of whether
        # .to(DEVICE) is a no-op (tensor already on DEVICE) or an actual copy
        x = feat_all[idx_in_fold].to(DEVICE).detach().clone().requires_grad_(True)
        c = clin_all[idx_in_fold].to(DEVICE)

        _, hg = model_grad(x, ei_r, ew_r, c)      # hg: (n_nodes, 64)

        path_emb   = hg[path_idx_t].mean(dim=0)   # (64,)
        path_score = path_emb.pow(2).sum()        # scalar (~= ||emb||^2)

        model_grad.zero_grad(set_to_none=True)    # avoid accumulating param .grad
        path_score.backward()

        with torch.no_grad():
            grad_x = x.grad[path_idx_t].abs().sum(dim=1)   # (n_path_genes,)
            for gene_i, g_idx in enumerate(path_idx):
                gene_grads[g_idx].append(grad_x[gene_i].item())

    gene_importance = {
        ppi_genes[g_idx]: float(np.mean(grads))
        for g_idx, grads in gene_grads.items()
        if grads
    }
    within_pathway_importance[pid] = gene_importance

del model_grad
torch.cuda.empty_cache()

n_empty = sum(1 for v in within_pathway_importance.values() if not v)
print(f"\nPathways processed : {len(within_pathway_importance)}  ({n_empty} with no usable patients)")


Using 30 high-risk test patients from seed42_fold0
Computing within-pathway gene importance for 31 pathways...


Pathways: 100%|██████████| 31/31 [00:19<00:00,  1.60it/s]


Pathways processed : 31  (0 with no usable patients)


---
## Cell 6 — Top Driver Genes Per Pathway + Save


In [6]:
print("Top 3 driver genes per significant pathway:")
print(f"{'Pathway':<45}  {'HR':>6}  {'Top genes (gradient importance)'}")
print("─" * 85)
for _, row in sig_pathways.sort_values('padj').head(15).iterrows():
    pid  = row['reactome_id']
    name = row['pathway_name'][:42]
    hr   = row['hr']
    gene_imp = within_pathway_importance.get(pid)
    if not gene_imp:
        continue
    top3 = sorted(gene_imp.items(), key=lambda x: -x[1])[:3]
    top3_str = ', '.join([f"{g}({v:.3f})" for g, v in top3])
    print(f"{name:<45}  {hr:>6.3f}  {top3_str}")

# Save full results
rows = []
for pid, gene_imp in within_pathway_importance.items():
    for gene, imp in sorted(gene_imp.items(), key=lambda x: -x[1]):
        rows.append({
            'reactome_id'  : pid,
            'pathway_name' : pathway_name_map.get(pid, ''),
            'gene_symbol'  : gene,
            'gradient_imp' : imp,
        })
wp_df = pd.DataFrame(rows)
wp_path = TABLE_DIR / 'within_pathway_gene_importance.csv'
wp_df.to_csv(wp_path, index=False)
print(f"\nSaved -> {wp_path}  ({len(wp_df):,} gene-pathway rows)")


Top 3 driver genes per significant pathway:
Pathway                                            HR  Top genes (gradient importance)
─────────────────────────────────────────────────────────────────────────────────────
Regulation of cholesterol biosynthesis by       0.682  CYP51A1(17.409), HMGCR(2.008), SCD(1.738)
SUMOylation                                     1.388  FOXL2(2.364), NR5A1(1.663), PML(1.589)
Citric acid cycle (TCA cycle)                   0.716  IDH2(2.662), ACAT1(2.419), NNT(2.026)
Regulation of MITF-M-dependent genes invol      1.339  CCND1(2.979), CDKN2A(2.449), LEF1(2.031)
Transcriptional regulation by RUNX3             1.313  PSMA2(1.402), CCND1(1.002), CCN2(0.876)
Aerobic respiration and respiratory electr      0.743  ATP5F1A(4.226), ATP5F1B(3.727), HIGD1A(2.334)
Activation of the mRNA upon binding of the      1.297  EIF4EBP1(8.621), EIF4A1(6.333), RPS14(0.807)
Specification of primordial germ cells          1.288  TFAP2C(1.540), CXCR4(1.494), EOMES(1.469)
Influenza 

---
## Cell 7 — Global Top-60 Genes by Gradient Importance (All Pathways Pooled)

`within_pathway_importance` gives a gradient score per gene *within each pathway* it
belongs to, so a gene appearing in several significant pathways has several scores.
To get one overall ranked list:

- **`gene_score`** = the gene's **max** gradient importance across all pathways it
  appears in (its strongest observed driving signal on any pathway).
- `mean_gradient_imp` and `n_pathways` are kept alongside for context (a high max
  from a single pathway reads differently than a consistently high score across many).

Sorted high → low, top 60 printed and saved to
`top60_genes_by_gradient.csv`.


In [7]:
# ── Pool gene gradient importance across all pathways, take global top 60 ────
from collections import defaultdict

gene_scores_all = defaultdict(list)   # gene_symbol -> list of gradient_imp (one per pathway it's in)

for pid, gene_imp in within_pathway_importance.items():
    for gene, imp in gene_imp.items():
        gene_scores_all[gene].append(imp)

gene_summary_rows = []
for gene, scores in gene_scores_all.items():
    gene_summary_rows.append({
        'gene_symbol'       : gene,
        'gene_score'        : float(np.max(scores)),   # ranking key: strongest signal across pathways
        'mean_gradient_imp' : float(np.mean(scores)),
        'n_pathways'        : len(scores),
    })

gene_summary_df = (
    pd.DataFrame(gene_summary_rows)
    .sort_values('gene_score', ascending=False)
    .reset_index(drop=True)
)

top60_genes = gene_summary_df.head(60).copy()
top60_genes.insert(0, 'rank', range(1, len(top60_genes) + 1))

print(f"Total unique genes across all significant pathways : {len(gene_summary_df):,}")
print(f"\nTop 60 genes by gradient importance (high -> low):")
print(f"{'Rank':>4}  {'Gene':<15}  {'Gradient score':>14}  {'Mean':>8}  {'#Pathways':>9}")
print("─" * 60)
for _, r in top60_genes.iterrows():
    print(f"{int(r['rank']):>4}  {r['gene_symbol']:<15}  {r['gene_score']:>14.4f}  "
          f"{r['mean_gradient_imp']:>8.4f}  {int(r['n_pathways']):>9}")

top60_path = TABLE_DIR / 'top60_genes_by_gradient.csv'
top60_genes.to_csv(top60_path, index=False)
print(f"\nSaved -> {top60_path}  ({len(top60_genes)} genes)")


Total unique genes across all significant pathways : 230

Top 60 genes by gradient importance (high -> low):
Rank  Gene             Gradient score      Mean  #Pathways
────────────────────────────────────────────────────────────
   1  ANGPTL4                 28.9730   28.9730          1
   2  CYP51A1                 17.4092   17.4092          1
   3  CGA                     16.2408   16.2408          1
   4  KCTD1                   13.0599   13.0599          1
   5  PTK6                    11.5584   10.1814          2
   6  DLX5                    11.0538    7.8762          2
   7  FOXH1                   10.9835   10.9835          1
   8  AXIN2                   10.5344   10.5344          1
   9  ZIC1                     9.4731    9.4731          1
  10  ALDOB                    9.3359    9.3359          1
  11  EIF4EBP1                 8.6206    8.6206          1
  12  EIF4A1                   6.3329    4.1520          2
  13  GSC                      6.3274    6.3274          1
  14